# Phase 1: Data Processing & Exploratory Data Analysis

This notebook handles the initial data pipeline and exploration stages for the retail demand forecasting project. The objective is to ingest the raw dataset, resolve data quality issues, and uncover baseline demand drivers before moving to time series forecasting.

### Pipeline Objectives
1. **Data Cleaning & Validation:**
   * Standardize data types (specifically datetime casting).
   * Handle missing values and remove duplicate records.
   * Detect and flag anomalies (e.g., negative inventory or pricing errors).
2. **Exploratory Data Analysis (EDA):**
   * Assess high-level sales metrics and temporal trends.
   * Identify operational bottlenecks (stockout vs. overstock frequencies across regions and categories).
   * Analyze external demand drivers, including promotional impact and weather conditions.
3. **Artifact Generation:**
   * Export the processed dataset as `cleaned_retail_data.csv`. This acts as an immutable, clean handoff file for the predictive modeling phase in Notebook 02.

### 1. Initial Data Inspection
First, we load the raw dataset to understand its structure. The goal here is to identify the total number of records, verify the data types assigned to each column, and get a baseline count of any missing values before applying transformations.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [3]:
import pandas as pd

# Reading from our "cloud storage" (Google Drive)
file_path = '/content/drive/MyDrive/Retail-Demand-Forecasting-Sales-Analysis/data/raw/retail_store_inventory.csv'
df = pd.read_csv(file_path)

# 1. Check the dimensions (rows and columns)
print(f"Dataset Shape: {df.shape}\n")

# 2. Check the data types and look for missing values structurally
print("--- Dataset Info ---")
df.info()

# 3. Check exact count of missing values per column
print("\n--- Missing Values Count ---")
print(df.isnull().sum())

# 4. Peek at the first 5 rows to see what the actual data looks like
print("\n--- First 5 Rows ---")
display(df.head())

Dataset Shape: (73100, 15)

--- Dataset Info ---
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 73100 entries, 0 to 73099
Data columns (total 15 columns):
 #   Column              Non-Null Count  Dtype  
---  ------              --------------  -----  
 0   Date                73100 non-null  object 
 1   Store ID            73100 non-null  object 
 2   Product ID          73100 non-null  object 
 3   Category            73100 non-null  object 
 4   Region              73100 non-null  object 
 5   Inventory Level     73100 non-null  int64  
 6   Units Sold          73100 non-null  int64  
 7   Units Ordered       73100 non-null  int64  
 8   Demand Forecast     73100 non-null  float64
 9   Price               73100 non-null  float64
 10  Discount            73100 non-null  int64  
 11  Weather Condition   73100 non-null  object 
 12  Holiday/Promotion   73100 non-null  int64  
 13  Competitor Pricing  73100 non-null  float64
 14  Seasonality         73100 non-null  object 
dtypes: f

,Date,Store ID,Product ID,Category,Region,Inventory Level,Units Sold,Units Ordered,Demand Forecast,Price,Discount,Weather Condition,Holiday/Promotion,Competitor Pricing,Seasonality
0,2022-01-01,S001,P0001,Groceries,North,231,127,55,135.47,33.50,20,Rainy,0,29.69,Autumn
1,2022-01-01,S001,P0002,Toys,South,204,150,66,144.04,63.01,20,Sunny,0,66.16,Autumn
2,2022-01-01,S001,P0003,Toys,West,102,65,51,74.02,27.99,10,Sunny,1,31.32,Summer
3,2022-01-01,S001,P0004,Toys,North,469,61,164,62.18,32.72,10,Cloudy,1,34.74,Autumn
4,2022-01-01,S001,P0005,Electronics,East,166,14,135,9.26,73.64,0,Sunny,0,68.95,Summer


### 2. Data Type Formatting and Deduplication
The dataset has zero missing values. The immediate priority is converting the `Date` column from a string (`object`) to a proper datetime format, which is a strict requirement for time series forecasting. Following this, we will remove any duplicate rows and inspect the categorical features for inconsistent naming conventions.

In [4]:
# 1. Convert 'Date' to datetime format
df['Date'] = pd.to_datetime(df['Date'])

# 2. Check for and drop exact duplicate rows
duplicates_count = df.duplicated().sum()
print(f"Exact duplicate rows found: {duplicates_count}")
if duplicates_count > 0:
    df = df.drop_duplicates()
    print(f"Dataset Shape after dropping duplicates: {df.shape}")

# 3. Check categorical consistency to look for typos or mismatched cases
print("\n--- Categorical Unique Values ---")
categorical_cols = ['Category', 'Region', 'Weather Condition', 'Seasonality']
for col in categorical_cols:
    print(f"{col}:")
    print(df[col].unique())
    print("-" * 20)

# 4. Verify the Date conversion
print("\n--- Updated Date Info ---")
print(f"Date dtype: {df['Date'].dtype}")
print(f"Date range: {df['Date'].min().date()} to {df['Date'].max().date()}")

Exact duplicate rows found: 0

--- Categorical Unique Values ---
Category:
['Groceries' 'Toys' 'Electronics' 'Furniture' 'Clothing']
--------------------
Region:
['North' 'South' 'West' 'East']
--------------------
Weather Condition:
['Rainy' 'Sunny' 'Cloudy' 'Snowy']
--------------------
Seasonality:
['Autumn' 'Summer' 'Winter' 'Spring']
--------------------

--- Updated Date Info ---
Date dtype: datetime64[ns]
Date range: 2022-01-01 to 2024-01-01


### 3. Outlier and Logical Check
Before finalizing the dataset, we must ensure the numeric values represent physical reality. We are checking the summary statistics to verify that there are no negative values for inventory, sales, or pricing, and to identify any extreme outliers that could distort the time series forecasting model later.

In [5]:
# 1. Generating summary statistics for numeric columns
print("--- Numeric Summary Statistics ---")
numeric_cols = ['Inventory Level', 'Units Sold', 'Units Ordered', 'Demand Forecast', 'Price', 'Competitor Pricing']
display(df[numeric_cols].describe().round(2))

# 2. Check for logical impossibilities (negative inventory, negative sales, zero or negative price)
print("\n --- Logical Validation Checks ---")
negative_inventory = (df['Inventory Level'] < 0).sum()
negative_sales = (df['Units Sold'] < 0).sum()
negative_pricing = (df['Price'] < 0).sum()

print(f"Rows with Negative Inventory: {negative_inventory}")
print(f"Rows with Negative Units Sold: {negative_sales}")
print(f"Rows with Negative Pricing: {negative_pricing}")

--- Numeric Summary Statistics ---


,Inventory Level,Units Sold,Units Ordered,Demand Forecast,Price,Competitor Pricing
count,73100.00,73100.00,73100.00,73100.00,73100.00,73100.00
mean,274.47,136.46,110.00,141.49,55.14,55.15
std,129.95,108.92,52.28,109.25,26.02,26.19
min,50.00,0.00,20.00,-9.99,10.00,5.03
25%,162.00,49.00,65.00,53.67,32.65,32.68
50%,273.00,107.00,110.00,113.02,55.05,55.01
75%,387.00,203.00,155.00,208.05,77.86,77.82
max,500.00,499.00,200.00,518.55,100.00,104.94



 --- Logical Validation Checks ---
Rows with Negative Inventory: 0
Rows with Negative Units Sold: 0
Rows with Negative Pricing: 0


### 4. Correcting Anomalies and Exporting
The summary statistics revealed a minimum `Demand Forecast` value of -9.99. Since negative demand is a logical impossibility, we will cap these values at 0. Finally, we will export this validated DataFrame as a CSV, completing the data cleaning pipeline.

In [6]:
# 1. Fixing the negative demand forecast values by capping them at 0
df['Demand Forecast'] = df['Demand Forecast'].clip(lower=0)
print(f"Minimum Demand Forecast after fix: {df['Demand Forecast'].min()}")

# 2. Exporting the clean dataset
export_path = 'cleaned_retail_data.csv'
df.to_csv(export_path, index=False)
print(f"Cleaned dataset exported to: {export_path}")

Minimum Demand Forecast after fix: 0.0
Cleaned dataset exported to: cleaned_retail_data.csv
